In [ ]:
CONNECTION = "ngrok"

ZROK = "Ce2OF9R0ANCV"
NGROK = "2a39vbvEM8fO1aj5h7p5VuibQvK_3SACZDUBhgmWb9cJcUqmA"

In [ ]:
import os
import subprocess
import threading
import time
import socket
import urllib

def run_command(cmd, printout=False):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, 
                        stderr=subprocess.PIPE, text=True, bufsize=1)
    stdout, stderr = p.communicate()
    if printout:
        if stdout:
            print(stdout)
        if stderr:
            print(stderr)
    return p

In [ ]:
!pip install fastapi uvicorn > /dev/null

import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  #  Замените на адрес сервиса анализа, если знаете
    allow_methods=["*"],
    allow_headers=["*"],
    allow_credentials=True,
)

@app.get("/")
async def root():
    return {"message": "Hello World"}

@app.post("/test")
async def test():
    return {
        "status": "ok"
    }

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server)
server_thread.start()

## Zrok

In [ ]:
def install_zrok():
    global ZROK
    
    # os.chdir('/tmp')
    get_ipython().system('wget -q https://github.com/openziti/zrok/releases/download/v2.0.0-rc5/zrok_2.0.0-rc5_linux_amd64.tar.gz')
    get_ipython().system('tar -xvzf zrok_2.0.0-rc5_linux_amd64.tar.gz')
    get_ipython().system('chmod +x zrok2')
    print("🔐 Enabling zrok environment...")
    cmd = f"./zrok2 enable {ZROK}"
    p = run_command(cmd)
    if p.returncode == 0:
        print("✅ zrok enabled successfully!")
    else:
        if 'you already have an enabled environment' in p.stderr:
            print("✅ zrok is already enabled!")
        else:
            print(f"❌ Error enabling zrok: {p.stderr}")


def zrok_thread(port=8000):
    cmd = f"./zrok2 share public localhost:{port} --headless"
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, 
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
    print('====== zrok enable =====')
    for line in p.stdout:
        print('[ZROK]:', line.strip())
        if 'shares.zrok.io' in line:
            import re
    
            m = re.search(r'(?im)\b([a-z0-9-]+\.shares\.zrok\.io)\b', line)
            url = m.group(1) if m else None
            
            msg = f'Zrok URL: {url}'
            border = '        +' + '-' * (len(msg) + 2) + '+'
            print()
            print(border)
            print(f".       | {msg} |")
            print(border)
            print()
        if '[ERROR]' in line:
            msg = 'Zrok Error: Cannot create public link. Visit https://api.zrok.io and delete some environments.'
            border = '        +' + '-' * (len(msg) + 2) + '+'
            print(border)
            print(f".       | {msg} |")
            print(border)

In [ ]:
ZROK

In [ ]:
!./zrok2 enable Ce2OF9R0ANCV

In [ ]:
install_zrok()

In [ ]:
zrok_thread()

## Ngrok

In [ ]:
def install_ngrok():
    get_ipython().system('pip install -q pyngrok')
    print("pyngrok installed!")

def ngrok_thread(port=8000):
    global ZROK
    
    print("\n Caution! Ngrok requires VPN in Russia!")
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK)
    http_tunnel = ngrok.connect(port)
    print(f'ngrok public URL: {http_tunnel.public_url}')

## localtunnel

In [ ]:
def install_localtunnel():
    get_ipython().system('npm install -g localtunnel')
    print("localtunnel installed!")

def localtunnel_thread(port=8000):
    print("Tunnel Password:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
    p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE)
    for line in p.stdout:
        print(line.decode(), end='')

## cloudflared

In [ ]:
def install_cloudflared():
    get_ipython().system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb')
    get_ipython().system('dpkg -i cloudflared-linux-amd64.deb')
    print("cloudflared installed!")

def cloudflared_thread(port=8000):
    get_ipython().system(f'cloudflared tunnel --url http://localhost:{port}')

## pinggy

In [ ]:
def pinggy_thread(port=8000):
    print("\n🌐 Starting Pinggy tunnel...")
    print("\n Caution! Pinggy requires VPN in Russia!")
    print("⏳ Waiting for Pinggy URL...\n")
    
    # Создаем SSH конфиг
    ssh_config = """
Host pinggy
    HostName a.pinggy.io
    Port 443
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ServerAliveInterval 30
    """
    
    get_ipython().system('mkdir -p ~/.ssh')
    with open(os.path.expanduser('~/.ssh/config'), 'w') as f:
        f.write(ssh_config)
    
    # Запуск pinggy
    cmd = f'ssh -R0:localhost:{port} pinggy'
    
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, 
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
    
    url_found = False
    for line in p.stdout:
        print('[PINGGY]:', line.strip())
        
        if not url_found and ('http://' in line or 'https://' in line):
            import re
            # Ищем URL формата http://xxxxx.a.pinggy.link
            urls = re.findall(r'https?://[a-zA-Z0-9\-\.]+\.pinggy\.[a-z]+', line)
            if urls:
                url = urls[0]
                msg = f'Pinggy URL: {url}'
                border = '        +' + '-' * (len(msg) + 2) + '+'
                print()
                print(border)
                print(f".       | {msg} |")
                print(border)
                print()
                url_found = True

In [ ]:
pinggy_thread()

In [ ]:
print("CONNECTION:", CONNECTION)

if CONNECTION == "ngrok":
    install_ngrok()
    threading.Thread(target=ngrok_thread, daemon=True).start()
elif CONNECTION == "zrok":
    install_zrok()
    threading.Thread(target=zrok_thread, daemon=True).start()
elif CONNECTION == "cloudflared":
    install_cloudflared()
    threading.Thread(target=cloudflared_thread, daemon=True).start()
elif CONNECTION == "localtunnel":
    install_localtunnel()
    threading.Thread(target=localtunnel_thread, daemon=True).start()
elif CONNECTION == "pinggy": 
    threading.Thread(target=pinggy_thread, daemon=True).start()
else:
    print(f"❌ Unsupported connection method: {connection.value}")

In [ ]:
zrok_token = "L1ZILuoDmhCx" # v9pUupyOfC1x, L1ZILuoDmhCx

In [ ]:
! rm -rf /opt/conda/lib/python3.10/site-packages/aiohttp-3.9.1.dist-info

In [ ]:
!curl -sSf https://get.openziti.io/install.bash | sudo bash -s zrok

In [ ]:
!zrok version

In [ ]:
!zrok enable L1ZILuoDmhCx

In [ ]:
import subprocess

def run_command(cmd, printout=False):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, 
                        stderr=subprocess.PIPE, text=True, bufsize=1)
    stdout, stderr = p.communicate()
    if printout:
        if stdout:
            print(stdout)
        if stderr:
            print(stderr)
    return p

In [ ]:
import os

def install_zrok(ZROK):
    os.chdir('/tmp')
    get_ipython().system('curl -sSf https://get.openziti.io/install.bash | sudo bash -s zrok')
    print("🔐 Enabling zrok environment...")
    cmd = f"zrok enable {ZROK}"
    p = run_command(cmd)
    if p.returncode == 0:
        print("✅ zrok enabled successfully!")
    else:
        if 'you already have an enabled environment' in p.stderr:
            print("✅ zrok is already enabled!")
        else:
            print(f"❌ Error enabling zrok: {p.stderr}")

def zrok_thread(port):
    cmd = f"zrok share public localhost:{port} --headless"
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, 
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
    print('====== zrok enable =====')
    for line in p.stdout:
        print('[ZROK]:', line.strip())
        if 'share.zrok.io' in line:
            import re
            url = re.search(r'https?://[^\s"]+', line).group()
            msg = f'Access ComfyUI via zrok URL: {url}'
            border = '        +' + '-' * (len(msg) + 2) + '+'
            print()
            print(border)
            print(f".       | {msg} |")
            print(border)
            print()
        if '[ERROR]' in line:
            msg = 'Zrok Error: Cannot create public link. Visit https://api.zrok.io and delete some environments.'
            border = '        +' + '-' * (len(msg) + 2) + '+'
            print(border)
            print(f".       | {msg} |")
            print(border)

def clean_up(conn_type):
    if conn_type == "zrok":
        run_command('zrok disable', printout=True)

In [ ]:
!zrok enable "L1ZILuoDmhCx"

In [ ]:
import threading

install_zrok(zrok_token)

threading.Thread(target=zrok_thread, daemon=True, args=(port,)).start()

In [ ]:
from kaggle_secrets import UserSecretsClient
import subprocess

token = zrok_token
subprocess.run(["/kaggle/working/bin/zrok", "enable", token], check=True)

In [ ]:
!mkdir /kaggle/working/zrok
%cd /kaggle/working/zrok
!wget -q https://github.com/openziti/zrok/releases/download/v1.1.11/zrok_1.1.11_linux_amd64.tar.gz
!tar -xvf ./zrok*.gz 
!chmod a+x /kaggle/working/zrok/zrok > /dev/null

!/kaggle/working/zrok/zrok enable $zrok_token > /dev/null
%cd /kaggle/working

In [ ]:
import subprocess
import time
import re
import requests

def process_url(url):
    print("Получен url:", url)

# Запускаем zrok асинхронно и перенаправляем вывод в файл
zrok_process = subprocess.Popen(
    ["/kaggle/working/zrok/zrok", "share", "public", "http://localhost:8000", "--headless"],
    stdout=open("zrok_output.txt", "w"),
    stderr=subprocess.STDOUT
)

# Ожидаем появления URL в файле
while True:
    with open("zrok_output.txt", "r") as f:
        output = f.read()
        match = re.search(r"https://[\w]+.share.zrok.io", output)
        if match:
            process_url(match.group())
            break  # Выходим из цикла после нахождения URL
    time.sleep(1)  # Проверяем файл каждую секунду

In [ ]:
import requests

url = "https://fyekid9yl6vh.share.zrok.io/test"

payload = {}

headers = {"Content-Type": "application/json"}

response = requests.post(url, headers=headers, params=payload)

print(response.status_code, response.text)

In [ ]:
import urllib.request
import os
import sys
import tarfile
import json
import subprocess
import platform

class Zrok:
    def __init__(self, token: str, name: str = None):
        """Initialize Zrok instance with API token and optional environment name.
        
        Args:
            token (str): Zrok API token for authentication
            name (str, optional): Name/description for the zrok environment. Defaults to None.
        """
        if token.startswith('<') and token.endswith('>'):
            raise ValueError("Please provide an actual your zrok token")
        
        self.token = token
        self.name = name
        self.base_url = "https://api-v1.zrok.io/api/v1"

    def get_env(self):
        """Get overview of all zrok environments using HTTP API.

        This method uses HTTP API to retrieve environments even when zrok enable command fails.
        
        Returns:
            dict: Overview data containing environments information
            None: If the API call fails or no environments exist
        """
        req = urllib.request.Request(
            url=f"{self.base_url}/overview",
            headers={"x-token": self.token},
        )

        with urllib.request.urlopen(req) as response:
            status = response.getcode()
            data = response.read().decode('utf-8')
            data = json.loads(data) 

        if status != 200:
            print(f"Error: {status}")
            raise Exception("zrok API overview error")
        
        return data['environments']

    def find_env(self, name: str):
        """Find a specific environment by its name.
        
        Args:
            name (str): Name/description of the environment to find (case-insensitive)
        
        Returns:
            dict: Environment information if found
            None: If no environment matches the given name
        """
        overview = self.get_env()
        if overview is None:
            return None

        for item in overview:
            env = item["environment"]
            if env["description"].lower() == name.lower():
                return item
            
        return None

    def delete_environment(self, zId: str):
        """Delete a zrok environment by its ID.
        
        Args:
            zid (str): The environment ID to delete
        
        Returns:
            bool: True if the environment was successfully deleted, False otherwise
        """
        headers = {
            "x-token": self.token,
            "Accept": "*/*",
            "Content-Type": "application/zrok.v1+json"
        }
        payload = {
            "identity": zId
        }
        
        data_bytes = json.dumps(payload).encode('utf-8')
        
        req = urllib.request.Request(f"{self.base_url}/disable", headers=headers, data=data_bytes, method="POST")
        with urllib.request.urlopen(req) as response:
            status = response.getcode()

        if status != 200:
            raise Exception("Failed to delete environment")

        return True

    def enable(self, name: str = None):
        """Enable zrok with the specified environment name.
        
        This method runs the 'zrok enable' command with the provided token and
        environment name. It will create a new environment if one doesn't exist.
        
        Args:
            name (str, optional): Name/description for the zrok environment.
                                 If not provided, uses the name from initialization.
            
        Raises:
            RuntimeError: If enable command fails
        """
        env_name = name if name is not None else self.name
        if env_name is None:
            raise ValueError("Environment name must be provided either during initialization or when calling enable()")
        
        subprocess.run(["zrok", "enable", self.token, "-d", env_name], check=True)

    def disable(self, name: str = None):
        """Disable zrok.
        
        This function executes the zrok disable command to delete the environment stored in the local file ~/.zrok/environment.json,
        and additionally removes any environments that could not be deleted through HTTP communication.
        
        Args:
            name (str, optional): Name/description for the zrok environment.
                                If not provided, uses the name from initialization.
        """
        env_name = name if name is not None else self.name

        # Delete the ~/.zrok/environment.json file
        try:
            subprocess.run(["zrok", "disable"], check=True)
        except Exception as e:
            print(e)
            print("zrok already disable")

        # Delete environment via HTTP communication even if zrok is not enabled
        env = self.find_env(env_name)
        if env is not None:
            self.delete_environment(env['environment']['zId'])

    @staticmethod
    def install():
        """Install the latest version of zrok.
        
        This method:
        1. Downloads the latest zrok release from GitHub
        2. Extracts the binary to /usr/local/bin/
        3. Verifies the installation
        """
        # Check if running on Windows
        if platform.system() != 'Linux':
            raise Exception("This script only works on Linux. For other operating systems, \
                            please install zrok manually following the instructions at https://docs.zrok.io/docs/guides/install/")

        print("Downloading latest zrok release")
        # Get latest release info
        response = urllib.request.urlopen("https://api.github.com/repos/openziti/zrok/releases/latest")
        data = json.loads(response.read())
        
        # Find linux_amd64 tar.gz download URL
        download_url = None
        for asset in data["assets"]:
            if "linux_amd64.tar.gz" in asset["browser_download_url"]:
                download_url = asset["browser_download_url"]
                break
        
        if not download_url:
            raise FileNotFoundError("Could not find zrok download URL for linux_amd64")
        
        # Download zrok
        urllib.request.urlretrieve(download_url, "zrok.tar.gz")
        
        print("Extracting Zrok")
        with tarfile.open("zrok.tar.gz", "r:gz") as tar:
            tar.extractall("/usr/local/bin/")
        os.remove("zrok.tar.gz")

        # Check if zrok is installed correctly
        if not Zrok.is_installed():
            raise RuntimeError("Failed to verify zrok installation")
        
        print("Successfully installed zrok")

    @staticmethod
    def is_installed():
        """Check if zrok is installed and accessible.
        
        Returns:
            bool: True if zrok is installed and can be executed, False otherwise
        """
        try:
            subprocess.run(["zrok", "version"], check=True)
            return True
        except (subprocess.CalledProcessError, FileNotFoundError):
            return False

    @staticmethod
    def is_enabled() -> bool:
        """Check if zrok is enabled.
        
        Returns:
            bool: True if zrok is enabled (Account Token and Ziti Identity are set), False otherwise
        """
        try:
            result = subprocess.run(
                ["zrok", "status"],
                capture_output=True,
                text=True,
                check=True
            )
            # Check if both Account Token and Ziti Identity are set
            return "Account Token  <<SET>>" in result.stdout and "Ziti Identity  <<SET>>" in result.stdout
        except subprocess.CalledProcessError:
            return False
        except FileNotFoundError:
            return False

In [ ]:
import subprocess
import argparse
import string
import random

def generate_random_password(length=16):
    characters = (string.ascii_letters + string.digits + "!@#$%^*()-_=+{}[]<>.,?")
    return ''.join(random.choices(characters, k=length))

ZROK = "Ce2OF9R0ANCV"
zrok = Zrok(ZROK, "kaggle_server")

if not Zrok.is_installed():
    Zrok.install()

zrok.disable()

In [ ]:
!zrok enable Ce2OF9R0ANCV -d kaggle_server

In [ ]:
import subprocess
import argparse
import string
import random

def generate_random_password(length=16):
    characters = (string.ascii_letters + string.digits + "!@#$%^*()-_=+{}[]<>.,?")
    return ''.join(random.choices(characters, k=length))


zrok = Zrok(ZROK, "kaggle_server")

if not Zrok.is_installed():
    Zrok.install()

zrok.disable()
# zrok.enable()

# Setup SSH server
print("Setting up SSH server...")
subprocess.run(["bash", "setup_ssh.sh"], check=True)


password = generate_random_password()
print(f"Setting password for root user: {password}")
subprocess.run(f"echo 'root:{password}' | sudo chpasswd", shell=True, check=True)